# 02 — Variáveis exógenas (seção 5.1 do v5)

**Lê:** `corte.json` (01) e o cache SGS em `dados_brutos/macro_cache/` (baixa da API do BCB só se faltar).  
**Grava:** `exogenas_semanais`, `macro_mensal`, `calendario_eventos`, `exogenas_listas.json`.

O cache fica fora da pasta da execução de propósito: o BCB revisa séries (ex.: IBC-Br), e reutilizar o mesmo arquivo garante que todas as execuções usam a mesma vintage. Registre a data do download no texto do TCC.

In [1]:
# === SETUP (idêntico em todos os notebooks; só muda NOTEBOOK) ===
NOTEBOOK = "02_exogenas"
RAIZ = "/content/drive/MyDrive/tcc_fluxo"      # Colab. Localmente: caminho da pasta do projeto (ou deixe None)

import os, sys
try:
    from google.colab import drive
    drive.mount("/content/drive")
    !pip install -q -r {RAIZ}/requirements.txt
except ImportError:                              # fora do Colab
    RAIZ = os.environ.get("TCC_RAIZ") or None
    if RAIZ is None:
        from pathlib import Path
        RAIZ = str(next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "config.yaml").is_file()))
sys.path.insert(0, f"{RAIZ}/src")
%load_ext autoreload
%autoreload 2

from IPython.display import display
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from tcc_fluxo import iniciar
C, A = iniciar(RAIZ, notebook=NOTEBOOK)

AVISO de reprodutibilidade: pandas=3.1.1; referência recomendada=2.2.3
AVISO de reprodutibilidade: numpy=2.5.4; referência recomendada=2.1.3
AVISO de reprodutibilidade: scikit-learn=1.9.1; referência recomendada=1.6.1
Notebook: 02_exogenas | execução: v6_modular | raiz: C:\Users\andre\OneDrive\Documentos\Projetos\Tccv2\tcc_fluxo_modular_v6


In [2]:
from tcc_fluxo import exogenas

corte = A.carregar_json("corte")
TEST_END = pd.Timestamp(corte["TEST_END"])
macro_long, macro_wide = exogenas.montar_macro(C.MACRO_SERIES, C.EXOGENOUS_API_START, C.EXOGENOUS_API_END,
                                               C.caminho("macro_cache"))
events = exogenas.calendario_eventos(range(C.PANEL_START.year - 1, TEST_END.year + 2))
exog, listas = exogenas.exogenas_semanais(events, macro_wide, C.MACRO_SERIES, C.PANEL_START, TEST_END,
                                          C.MACRO_LAG_MONTHS)
print(f"Macro: {len(C.MACRO_SERIES)} séries, {len(macro_wide)} meses, atraso de {C.MACRO_LAG_MONTHS} meses")
print(f"Calendário: {len(listas['CALENDAR_FEATURES'])} atributos, {len(events)} datas | exógenas: {len(listas['EXOGENOUS_FEATURES'])}")
display(exog.head())

Macro: 4 séries, 36 meses, atraso de 2 meses
Calendário: 22 atributos, 68 datas | exógenas: 26


,target_week,reference_week,macro_reference_month,macro_ipca,macro_ibcbr,macro_icc,macro_selic,cal_holiday_new_year,cal_holiday_carnival,cal_holiday_good_friday,cal_holiday_easter,cal_holiday_tiradentes,cal_holiday_labor_day,cal_holiday_corpus_christi,cal_holiday_independence,cal_holiday_children_day,cal_holiday_all_souls,cal_holiday_republic,cal_holiday_christmas,cal_commercial_consumer_day,cal_commercial_mothers_day,cal_commercial_valentines_day,cal_commercial_fathers_day,cal_commercial_black_friday,cal_holiday_count,cal_commercial_count,cal_any_holiday,cal_any_commercial,cal_event_count
0,2017-01-02,2016-12-26,2016-10-01,0.2600,92.7317,105.9800,1.0500,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000
1,2017-01-09,2017-01-02,2016-11-01,0.1800,92.6233,110.3000,1.0400,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000
2,2017-01-16,2017-01-09,2016-11-01,0.1800,92.6233,110.3000,1.0400,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000
3,2017-01-23,2017-01-16,2016-11-01,0.1800,92.6233,110.3000,1.0400,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000
4,2017-01-30,2017-01-23,2016-11-01,0.1800,92.6233,110.3000,1.0400,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000


In [3]:
A.salvar_df(exog, "exogenas_semanais")
A.salvar_df(macro_long, "macro_mensal")
A.salvar_df(events, "calendario_eventos")
cache_hash = {p.name: __import__("tcc_fluxo.io", fromlist=["x"]).sha256_arquivo(p)
              for p in sorted(C.caminho("macro_cache").glob("*.csv"))}
A.salvar_json({**listas, "fonte": C.MACRO_SOURCE, "series_sgs": C.MACRO_SERIES,
               "lag_macro_meses": C.MACRO_LAG_MONTHS, "cache_sha256": cache_hash,
               "n_meses_macro": len(macro_wide), "n_semanas_exogenas": len(exog), "n_eventos": len(events)},
              "exogenas_listas")
A.tabela(exog, "calendario_macro_semanal.csv")

WindowsPath('C:/Users/andre/OneDrive/Documentos/Projetos/Tccv2/tcc_fluxo_modular_v6/execucoes/v6_modular/saidas/calendario_macro_semanal.csv')

In [4]:
A.finalizar()

02_exogenas: leu 1 artefato(s), gravou 4.
